# tiny-gpt, from zero

You'll build a GPT, the same architecture as GPT-2 at toy scale, one piece at a time, and train it to write (bad) Shakespeare.
You start from "what is a token?" and end with a 10.8M-parameter model at **val loss < 1.6**.

### How to use this notebook
1. **Read** the markdown for a part: 🎯 goal, 🧠 idea, 🛠️ what to build.
2. **Write** your code in the cells marked `# ✍️ YOUR CODE`. Type it yourself; don't paste.
3. **Run** the ✅ check cell under it. Don't edit check cells. If one fails, its message says what's wrong.
4. Stuck for more than 10 minutes? Ask Knak for **a hint**, not the answer.

Kernel: **`.aiml` (Python 3.12)**. One part ≈ one pomodoro.
Restarted the kernel? Run all cells above where you are (*Run Above*) before continuing.

### Roadmap

| Part | What you build | You learn | Time |
|---|---|---|---|
| 0 | Setup | | 5 min |
| 1 | Tokenizer: `encode` / `decode` | text → numbers | 15 min |
| 2 | Dataset tensor + train/val split | why we hold data out | 10 min |
| 3 | `get_batch` | the next-token task; one window = T examples | 20 min |
| 4 | `BigramLM` + training loop | logits, softmax, cross-entropy, backprop, sampling | 45 min |
| 5 | The averaging trick | how tokens can look at the past, but not the future | 25 min |
| 6 | One attention head | query, key, value; √d scaling; the causal mask | 40 min |
| 7 | `CausalSelfAttention` (multi-head) | heads, fused QKV, `scaled_dot_product_attention` | 45 min |
| 8 | `MLP` | per-token computation, GELU, the 4× width | 15 min |
| 9 | `Block` | residual stream, LayerNorm, pre-norm | 25 min |
| 10 | `GPT` | embeddings + positions, the full forward pass, `generate` | 40 min |
| 11 | Real training run | AdamW, warm-up, clipping, bf16, overfitting | 45 min |
| 12 | Sampling + checkpoint | temperature, saving for Monday | 15 min |
| 13 | Notebook → `model.py` | the 5 tests in `test_model.py`, `train.py`, commit | 30 min |

About **6 hours** in total. Parts 0–4 are the foundation; if those make sense, the rest is incremental.

## Part 0 · Setup (5 min)

Imports, the device (your RTX 5070 if CUDA works) and a fixed random seed, so your numbers match the expected ranges. Just run it.

In [2]:
import math
import time
from dataclasses import dataclass
from pathlib import Path

import torch
import torch.nn as nn
from torch.nn import functional as F

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(1337)
print(f"torch {torch.__version__} on {device}" + (f" ({torch.cuda.get_device_name()})" if device == "cuda" else ""))

torch 2.14.1+cu132 on cuda (NVIDIA GeForce RTX 5070)


## Part 1 · Text to numbers: the tokenizer (15 min)

🎯 **Goal:** turn the Shakespeare text into a list of integers, and back.

🧠 **Idea.** A neural network only does arithmetic, so text has to become numbers first. A **tokenizer** splits text into **tokens** and gives each token an id.

- We use the simplest one possible: **every distinct character is a token**. Tiny Shakespeare has 65 distinct characters (letters, punctuation, space, newline), so the **vocabulary size is 65**.
- GPT-2 uses BPE instead, with about 50,000 tokens that are pieces of words. Fewer tokens per sentence and more meaning per token, but more machinery. Characters keep the focus on the model.

🛠️ **Build:**
1. `text`: the contents of `data/input.txt`. Print its length and the first 300 characters to see what you're working with.
2. `vocab`: the **sorted** list of distinct characters, and `vocab_size = len(vocab)`.
3. `stoi` (string → int) and `itos` (int → string): two dicts that map each character to its position in `vocab` and back.
4. `encode(s)`, which takes a string and returns a list of ints, and `decode(ids)`, which takes a list of ints and returns a string.

In [ ]:
text = Path("data/input.txt").read_text()
print(f"Loaded text from data/input.txt, length: {len(text)} characters")


vocab = sorted(list(set(text)))
vocab_size = len(vocab)
print(f"Unique characters: {vocab_size}")
print(f"Characters: {vocab}")# ✍️ YOUR CODE: text, vocab, vocab_size, stoi, itos, encode, decode

stoi = {ch: i for i, ch in enumerate(vocab)} # string to index mapping
itos = {i: ch for i, ch in enumerate(vocab)} # index to string mapping



Loaded text from data/input.txt, length: 1115394 characters
Unique characters: 65
Characters: ['\n', ' ', '!', '$', '&', "'", ',', '-', '.', '3', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']


In [4]:
# ✅ CHECK Part 1
assert len(text) == 1_115_394, f"text should be 1,115,394 characters, got {len(text):,}"
assert vocab_size == 65, f"expected 65 distinct characters, got {vocab_size}"
assert vocab == sorted(vocab), "vocab must be sorted, so ids are stable between runs"
assert itos[0] == "\n" and itos[1] == " ", "after sorting, id 0 is '\\n' and id 1 is ' '"
assert decode(encode("To be, or not to be")) == "To be, or not to be", "decode(encode(s)) must give s back"
assert encode("hi") == [stoi["h"], stoi["i"]]
print("✅ Part 1 passed:", encode("hii there"), "→", repr(decode(encode("hii there"))))

NameError: name 'chars' is not defined

## Part 2 · The dataset tensor and the train/val split (10 min)

🎯 **Goal:** the whole text as one long tensor of ids, split into a training part and a validation part.

🧠 **Idea.**
- PyTorch works on **tensors**, so encode all 1.1M characters into one 1-D `torch.long` tensor.
- **Why hold data out?** A 10M-parameter model can memorise 1 MB of text. Then the training loss keeps falling while the model learns nothing general.
  The **validation** set is text the model never trains on. Its loss is the honest score: it measures whether the model learned *Shakespeare*, not *this file*.
- Split **by position, not randomly**: the first 90% for training, the last 10% for validation. A random split would scatter the same scenes across both sets, and validation would look easier than it is.

🛠️ **Build:**
- `data`: `torch.tensor(encode(text), dtype=torch.long)`
- `n = int(0.9 * len(data))`, then `train_data` is everything before `n` and `val_data` everything from `n` on.

In [ ]:
# ✍️ YOUR CODE: data, train_data, val_data


In [ ]:
# ✅ CHECK Part 2
assert isinstance(data, torch.Tensor) and data.dtype == torch.long and data.ndim == 1, "data must be a 1-D torch.long tensor"
assert len(data) == len(text)
assert len(train_data) == int(0.9 * len(data)) and len(train_data) + len(val_data) == len(data)
assert torch.equal(torch.cat([train_data, val_data]), data), "train then val must be the data in its original order"
print(f"✅ Part 2 passed: {len(train_data):,} train ids, {len(val_data):,} val ids")

## Part 3 · The prediction task and batches (20 min)

🎯 **Goal:** understand exactly what the model is trained to do, and write `get_batch`.

🧠 **Idea.** A language model has one job: **given the text so far, predict the next token.**

Take a window of `block_size + 1` characters, say `block_size = 8`. It contains **8 training examples**, not 1:

```
window:  F i r s t _ C i t        (9 chars)
context "F"         → target "i"
context "Fi"        → target "r"
context "Fir"       → target "s"
...
context "First Ci"  → target "t"
```

- So `x` is the first 8 characters and `y` is the same 8 **shifted one to the left**: `y[t]` is the character that follows `x[0..t]`.
- Training on contexts of every length from 1 to `block_size` is also why generation can start from a single character.
- `block_size` is the **longest context** the model ever sees. The final model uses 256.

**Batches.** A GPU is fast at doing many independent things at once. So we take `batch_size` (B) random windows and stack them, which gives `x` and `y` of shape **(B, T)**, where T = `block_size`.

🛠️ **Build:**
1. **Look first:** take `x = train_data[:9]` and print the 8 *(context → target)* pairs above using `decode`. (No check for this; it's for your understanding.)
2. `get_batch(split, batch_size, block_size)`:
   - pick `train_data` or `val_data` by `split` (`"train"` / `"val"`);
   - draw `batch_size` random start positions with `torch.randint`. What is the largest valid start?
   - `x` = stack of `data[i : i+block_size]`, `y` = stack of `data[i+1 : i+block_size+1]`;
   - return both moved to `device`.

In [ ]:
# ✍️ YOUR CODE: print the 8 (context → target) pairs of the first window


In [ ]:
# ✍️ YOUR CODE: get_batch(split, batch_size, block_size) -> x, y


In [ ]:
# ✅ CHECK Part 3
xb, yb = get_batch("train", 4, 8)
assert xb.shape == (4, 8) and yb.shape == (4, 8), f"expected (4, 8) for both, got {tuple(xb.shape)} and {tuple(yb.shape)}"
assert str(xb.device).startswith(device), "move x and y to `device`"
assert torch.equal(yb[:, :-1], xb[:, 1:]), "y must be x shifted left by one: y[:, t] == x[:, t+1]"
for row_x, row_y in zip(xb.tolist(), yb.tolist()):
    assert decode(row_x + row_y[-1:]) in text, "each window must be a real contiguous piece of the text"
xv, _ = get_batch("val", 2, 8)
assert decode(xv[0].tolist()) in decode(val_data.tolist()), "split='val' must sample from val_data"
starts = {tuple(get_batch("train", 1, 8)[0][0].tolist()) for _ in range(20)}
assert len(starts) > 1, "windows should start at random positions"
print("✅ Part 3 passed. One batch:")
for row in xb.tolist():
    print("  ", repr(decode(row)))

## Part 4 · The simplest language model: a bigram (45 min)

🎯 **Goal:** a model that predicts the next character from **only the current one**, trained with a real training loop. Everything later reuses this loop.

🧠 **Idea: a lookup table.** For each current character, store 65 scores, one per possible next character. That's a 65 × 65 table.
`nn.Embedding(vocab_size, vocab_size)` is exactly that: row `i` is the scores for "what follows character `i`".

- **Logits** are the raw scores. **Softmax** turns them into probabilities: `p = exp(logits) / sum(exp(logits))`.
- **Loss: cross-entropy** = `−log p(correct next char)`, averaged over every position.
  - Perfect prediction gives 0.
  - A uniform guess over 65 characters gives `−log(1/65) = ln 65 ≈ 4.17`, so an untrained model should start near there.
  - Random initial weights aren't perfectly uniform, so expect slightly more, about 4.5–4.9.
- **Shapes:** `F.cross_entropy` wants logits as **(N, C)** and targets as **(N,)**. Ours are (B, T, C) and (B, T), so flatten B·T into N with `.view(B*T, C)` and `.view(B*T)`.

**Generation** (sampling): start from some ids `(B, T)`, then repeat `max_new_tokens` times:
1. run the model;
2. keep only the **last** position's logits, `(B, C)`;
3. softmax gives probabilities;
4. `torch.multinomial(probs, 1)` samples one id;
5. append it to the sequence with `torch.cat(..., dim=1)`.

🛠️ **Build 4a: `BigramLM(vocab_size)`**, an `nn.Module`:
- `__init__`: the embedding table.
- `forward(idx, targets=None)` returns `(logits, loss)`. `logits` is (B, T, vocab); `loss` is `None` when there are no targets.
- `generate(idx, max_new_tokens)` returns idx with the new tokens appended, shape (B, T + max_new_tokens).

In [ ]:
# ✍️ YOUR CODE: class BigramLM(nn.Module)


In [ ]:
# ✅ CHECK Part 4a
torch.manual_seed(1337)
_m = BigramLM(vocab_size).to(device)
_x, _y = get_batch("train", 4, 8)
_logits, _loss = _m(_x, _y)
assert _logits.shape == (4, 8, vocab_size), f"logits should be (B, T, vocab) = (4, 8, 65), got {tuple(_logits.shape)}"
assert _m(_x)[1] is None, "with no targets, loss must be None"
assert 4.0 < _loss.item() < 5.5, f"untrained loss should be a bit above ln(65) = 4.17, got {_loss.item():.2f}"
_out = _m.generate(torch.zeros(1, 1, dtype=torch.long, device=device), 30)
assert _out.shape == (1, 31), f"generate should append 30 tokens: expected (1, 31), got {tuple(_out.shape)}"
print(f"✅ Part 4a passed. Starting loss {_loss.item():.3f} (ln 65 = {math.log(65):.3f})")
print("Untrained sample:", repr(decode(_out[0].tolist())))

🧠 **The training loop.** Repeat for many steps:
1. `xb, yb = get_batch("train", ...)`
2. `logits, loss = model(xb, yb)`, the **forward** pass
3. `opt.zero_grad(set_to_none=True)` clears the old gradients. PyTorch *adds* new gradients to existing ones, so they must be cleared every step.
4. `loss.backward()` computes, for every weight, which direction lowers the loss (**backpropagation**).
5. `opt.step()` nudges every weight a little in that direction.

**Optimizer:** `torch.optim.AdamW(model.parameters(), lr=...)`. AdamW is plain gradient descent plus a per-weight adaptive step size; it's the default for transformers.

**Measuring progress:** the loss on one batch is noisy, so `estimate_loss` averages it over many batches, for both splits:
- call `model.eval()` first (it turns dropout off; that matters from Part 7);
- wrap it in `@torch.no_grad()`, since there's no backward pass, so it's faster and uses less memory;
- call `model.train()` again at the end.

🛠️ **Build 4b:**
1. `estimate_loss(model, batch_size, block_size, eval_iters=100)` returns `{"train": float, "val": float}`.
2. Train a fresh `BigramLM` on `device` for **3000 steps** with `batch_size=32`, `block_size=8`, `lr=1e-2`. Print both losses every 500 steps. Name the model `bigram`.
3. Print a 300-character sample from it.

Expect the loss to fall from about 4.7 to about **2.5** and stop there. The output will look *word-shaped* but be nonsense.

In [ ]:
# ✍️ YOUR CODE: estimate_loss(...)


In [ ]:
# ✍️ YOUR CODE: train `bigram` for 3000 steps, then print a 300-char sample


In [ ]:
# ✅ CHECK Part 4b
_l = estimate_loss(bigram, 32, 8)
assert set(_l) == {"train", "val"}, "estimate_loss must return a dict with 'train' and 'val'"
assert bigram.training, "estimate_loss must switch the model back to train mode at the end"
assert _l["val"] < 2.7, f"a trained bigram reaches about 2.5 val loss; got {_l['val']:.3f}. Did the loop call zero_grad, backward and step?"
print(f"✅ Part 4b passed: val loss {_l['val']:.3f}")
print("The bigram only ever sees ONE character of context. That is its ceiling. Next: let tokens look further back.")

## Part 5 · Letting tokens look at the past: the averaging trick (25 min)

🎯 **Goal:** the core mathematical trick behind attention, before any learning is involved.

🧠 **Idea.** To predict well, position *t* should use information from positions **0…t**, but **never from t+1 onward**: those hold the answer.
The crudest way to "gather the past" is to **average** the vectors at positions 0…t. You'll compute it three ways. The third is exactly the shape attention takes.

1. **Loops:** for every b and t, `x[b, :t+1].mean(dim=0)`.
2. **Matrix multiply:** a (T, T) weight matrix whose row t holds `1/(t+1)` in columns 0…t and 0 elsewhere (a lower-triangular matrix), then `wei @ x`. Use `torch.tril(torch.ones(T, T))`, and divide each row by its sum.
3. **Softmax over masked scores:**
   - start from `scores = torch.zeros(T, T)`;
   - set the "future" entries (where the `tril` matrix is 0) to `-inf` with `masked_fill`;
   - apply `softmax(dim=-1)`. Since `exp(-inf) = 0`, every row becomes the same uniform average as in version 2.

Version 3 is the key: **attention = version 3 with learned, data-dependent scores instead of zeros.**
Every token then decides how *much* to take from each earlier token, instead of taking an equal share.

In [ ]:
torch.manual_seed(1337)
B, T, C = 4, 8, 2
x = torch.randn(B, T, C)

In [ ]:
# ✍️ YOUR CODE: xbow1 (loops), xbow2 (tril matmul), xbow3 (masked softmax). All shaped (B, T, C)


In [ ]:
# ✅ CHECK Part 5
for name, v in [("xbow1", xbow1), ("xbow2", xbow2), ("xbow3", xbow3)]:
    assert v.shape == (B, T, C), f"{name} should be (B, T, C) = {(B, T, C)}, got {tuple(v.shape)}"
assert torch.allclose(xbow1[0, 0], x[0, 0]), "position 0 can only see itself"
assert torch.allclose(xbow1[0, -1], x[0].mean(dim=0)), "the last position averages the whole window"
assert torch.allclose(xbow1, xbow2, atol=1e-6), "the matmul version must match the loop version"
assert torch.allclose(xbow1, xbow3, atol=1e-6), "the masked-softmax version must match the loop version"
print("✅ Part 5 passed: three ways, one answer. Version 3 + learned scores = attention.")

## Part 6 · Self-attention, one head (40 min)

🎯 **Goal:** replace the uniform average with **learned, data-dependent** weights.

🧠 **Idea.** Every token produces three vectors, each through its own `nn.Linear(C, head_size, bias=False)`:

| Vector | Role | Analogy |
|---|---|---|
| **query** q | "what am I looking for?" | a search query |
| **key** k | "what do I contain?" | a document's title |
| **value** v | "what do I hand over if picked?" | the document's content |

- **Scores:** `q @ k.transpose(-2, -1)` gives a (B, T, T) matrix: how well each token's query matches every token's key.
- **Scale:** multiply the scores by `head_size ** -0.5`. Why: q·k is a sum of `head_size` products, so its variance grows with `head_size`. Large scores push the softmax towards one-hot, where its gradients vanish. The demo below shows this.
- **Causal mask:** set the future entries to `-inf`, exactly as in Part 5. Store the `tril` matrix with `self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))`. A buffer moves with `.to(device)` but isn't a trainable weight.
  Inside `forward`, use `self.tril[:T, :T]`, because T can be shorter than `block_size`.
- **Softmax** over the last dimension gives weights `wei`, (B, T, T), where each row sums to 1.
- **Output:** `wei @ v` gives (B, T, head_size).

This is **"self"**-attention because q, k and v all come from the same `x`. And it's the **only** place in a transformer where tokens exchange information.

In [ ]:
# Demo: why scale by 1/sqrt(head_size)? q and k have unit variance here.
for hs in (16, 64, 256):
    q, k = torch.randn(1000, hs), torch.randn(1000, hs)
    raw = (q * k).sum(-1)
    print(f"head_size {hs:3d}: var(q·k) = {raw.var():6.1f}   var(q·k / sqrt(hs)) = {(raw / hs ** 0.5).var():.2f}")
s = torch.tensor([0.1, -0.2, 0.3, 0.2])
print("softmax(s)      =", F.softmax(s, -1).numpy().round(3), " ← spread out")
print("softmax(s * 16) =", F.softmax(s * 16, -1).numpy().round(3), " ← nearly one-hot: tiny gradients")

In [ ]:
# ✍️ YOUR CODE: class Head(nn.Module) with __init__(self, n_embd, head_size, block_size) and forward(x) -> (B, T, head_size)


🧪 **Write your own test.** The one property attention must never break is **causality**: changing the *future* must not change the *past* outputs.

Write `test_head_is_causal()`:
1. build a `Head(32, 16, block_size=8)`;
2. make a random `x` of shape (1, 8, 32);
3. make `y` as a copy of `x` with positions 5, 6 and 7 replaced by new random values;
4. `assert` that the outputs at positions 0–4 are equal for `x` and `y` (`torch.allclose(..., atol=1e-6)`).

Then call it. This is how ML code gets tested: **you can't assert the exact numbers, so you assert properties** (shapes, invariances, sane starting values).

In [ ]:
# ✍️ YOUR CODE: def test_head_is_causal(): ... then call it


In [ ]:
# ✅ CHECK Part 6
torch.manual_seed(0)
_h = Head(32, 16, 8)
_x = torch.randn(2, 8, 32)
assert _h(_x).shape == (2, 8, 16), f"expected (B, T, head_size) = (2, 8, 16), got {tuple(_h(_x).shape)}"
assert _h(_x[:, :5]).shape == (2, 5, 16), "must work when T < block_size (slice tril to [:T, :T])"
_y = _x.clone(); _y[:, 5:] = torch.randn(2, 3, 32)
assert torch.allclose(_h(_x)[:, :5], _h(_y)[:, :5], atol=1e-6), "NOT causal: the future leaks into the past. Check the mask"
assert not torch.allclose(_h(_x)[:, 5:], _h(_y)[:, 5:]), "outputs at 5..7 should change when inputs 5..7 change"
assert "tril" in dict(_h.named_buffers()), "register the mask with register_buffer('tril', ...)"
assert sum(p.numel() for p in _h.parameters()) == 3 * 32 * 16, "three Linear(n_embd, head_size, bias=False) layers"
print("✅ Part 6 passed: one causal attention head")

## Part 7 · Multi-head attention, the efficient way (45 min)

🎯 **Goal:** the real `CausalSelfAttention` from `model.py` (TODOs 1 and 2).

🧠 **Idea.**
- **Several heads in parallel:** each head can learn a different relation, for example "the previous character", "the start of this word" or "who is speaking".
  With `n_head` heads of size `hs = C // n_head`, concatenating their outputs gives back width C.
- **One matmul, not 3 × n_head:** a single `nn.Linear(C, 3*C)` produces q, k and v for **all** heads at once.
  1. `.split(C, dim=2)` gives three tensors of (B, T, C).
  2. `.view(B, T, n_head, hs)` cuts each one into heads.
  3. `.transpose(1, 2)` gives **(B, nh, T, hs)**, so every head is an independent batch entry.
- **Output projection** `nn.Linear(C, C)`: after merging the heads back to (B, T, C), this mixes their results.
  To merge: `.transpose(1, 2).contiguous().view(B, T, C)`. `contiguous()` is needed because `view` requires memory laid out in order.
- **Dropout** randomly zeroes activations during training only. It fights overfitting on our small dataset. It's applied to the attention weights and to the output.
- **`F.scaled_dot_product_attention(q, k, v, is_causal=True, dropout_p=...)`** does your Part 6 math (scale, mask, softmax, @v) in one fused GPU kernel (FlashAttention).
  First you'll write it by hand for 4-D tensors and prove they match; then use the fast one.

The config below holds all the sizes. It's the same `GPTConfig` as in `model.py`.

In [ ]:
@dataclass
class GPTConfig:
    vocab_size: int = 65    # distinct characters
    block_size: int = 256   # longest context; also the size of the position table
    n_layer: int = 6        # how many Blocks are stacked
    n_head: int = 6         # heads per attention layer (head size = n_embd // n_head = 64)
    n_embd: int = 384       # width of every token vector (the residual stream)
    dropout: float = 0.2    # fraction of activations zeroed in training

# small config for the checks: runs on the CPU in a second, dropout off so results are deterministic
CFG = GPTConfig(vocab_size=65, block_size=16, n_layer=2, n_head=4, n_embd=32, dropout=0.0)

🛠️ **Build 7a:** `manual_attention(q, k, v)` for inputs shaped (B, nh, T, hs): scale, causal mask, softmax, `@ v`, the same as your `Head` but for 4-D tensors.
Build the mask from `torch.ones(T, T, device=q.device)`.

In [ ]:
# ✍️ YOUR CODE: def manual_attention(q, k, v) -> (B, nh, T, hs)


In [ ]:
# ✅ CHECK Part 7a
torch.manual_seed(0)
_q, _k, _v = (torch.randn(2, 4, 10, 8) for _ in range(3))
_mine = manual_attention(_q, _k, _v)
_ref = F.scaled_dot_product_attention(_q, _k, _v, is_causal=True)
assert _mine.shape == (2, 4, 10, 8)
assert torch.allclose(_mine, _ref, atol=1e-5), f"differs from PyTorch by {(_mine - _ref).abs().max():.2e}. Check the scale and the mask"
print("✅ Part 7a passed: your attention matches F.scaled_dot_product_attention")

🛠️ **Build 7b:** `CausalSelfAttention(cfg)`
- `__init__`:
  - `c_attn = nn.Linear(C, 3*C)` and `c_proj = nn.Linear(C, C)`;
  - store `n_head`, `n_embd` and the dropout rate;
  - `resid_dropout = nn.Dropout(dropout)`.
- `forward(x)`:
  1. project, split into q, k, v, and split the heads;
  2. `F.scaled_dot_product_attention(q, k, v, is_causal=True, dropout_p=self.dropout if self.training else 0.0)`;
  3. merge the heads;
  4. `c_proj`, then `resid_dropout`.

In [ ]:
# ✍️ YOUR CODE: class CausalSelfAttention(nn.Module)


In [ ]:
# ✅ CHECK Part 7b  (= test_attention_shape + test_attention_is_causal in test_model.py)
torch.manual_seed(0)
_a = CausalSelfAttention(CFG).eval()
_x = torch.randn(2, 10, CFG.n_embd)
assert _a(_x).shape == _x.shape, f"attention must keep the shape (B, T, C); got {tuple(_a(_x).shape)}"
_y = _x.clone(); _y[:, 5:] = torch.randn(2, 5, CFG.n_embd)
assert torch.allclose(_a(_x)[:, :5], _a(_y)[:, :5], atol=1e-6), "NOT causal: the future leaks into the past"
_n = sum(p.numel() for p in _a.parameters()); C = CFG.n_embd
assert _n in (4 * C * C + 4 * C, 4 * C * C), f"expected {4*C*C + 4*C} params (c_attn C→3C + c_proj C→C, with bias); got {_n}"
print(f"✅ Part 7b passed: multi-head causal self-attention, {_n:,} params")

## Part 8 · The feed-forward MLP (15 min)

🎯 **Goal:** the per-token "thinking" layer.

🧠 **Idea.**
- Attention is **communication**: tokens gather information from each other.
- The MLP is **computation**: each token, on its own, processes what it gathered. The same small network is applied to every position independently.
- Shape: `Linear(C, 4C)`, then `GELU`, then `Linear(4C, C)`, then `Dropout`.
- **4×** wider in the middle is the GPT convention, and it means the MLP holds about **2/3 of all the parameters**.
- **GELU** is a smooth ReLU: it lets small negative values through a little, so gradients don't die at exactly 0. GPT uses it.

🛠️ **Build:** `MLP(cfg)` with `forward(x)`, mapping (B, T, C) to (B, T, C). `nn.Sequential` is fine.

In [ ]:
# ✍️ YOUR CODE: class MLP(nn.Module)


In [ ]:
# ✅ CHECK Part 8
torch.manual_seed(0)
_mlp = MLP(CFG).eval(); C = CFG.n_embd
_x = torch.randn(2, 10, C)
assert _mlp(_x).shape == _x.shape
_n = sum(p.numel() for p in _mlp.parameters())
assert _n in (8 * C * C + 5 * C, 8 * C * C), f"expected {8*C*C + 5*C} params (C→4C→C with bias); got {_n}"
_y = _x.clone(); _y[:, 3] = torch.randn(2, C)
_changed = (_mlp(_x) - _mlp(_y)).abs().sum(dim=(0, 2)) > 0
assert _changed.tolist() == [t == 3 for t in range(10)], "the MLP must act on each position independently"
assert any(isinstance(m, nn.GELU) for m in _mlp.modules()), "use nn.GELU as the activation"
print(f"✅ Part 8 passed: per-token MLP, {_n:,} params")

## Part 9 · The transformer block: residuals and LayerNorm (25 min)

🎯 **Goal:** one transformer layer, stacked 6 times in the final model.

```
x = x + attn(ln_1(x))     # communicate
x = x + mlp(ln_2(x))      # compute
```

🧠 **Idea.**
- **Residual connection** (`x + f(x)`, from ResNet): each sublayer only *adds a correction* to a running vector called the **residual stream**.
  The gradient of `x + f(x)` with respect to `x` always contains a plain 1, so gradients flow straight down through all the layers. That's what makes deep stacks trainable.
- **LayerNorm** rescales **each token's vector** to mean 0 and variance 1, then applies a learned scale and shift (`nn.LayerNorm(C)` has 2·C parameters). It keeps activations in a healthy range layer after layer.
- **Pre-norm:** the LayerNorm goes *before* each sublayer, inside the residual branch. The original 2017 paper put it after (post-norm); GPT-2 moved it, and that trains more stably.

🛠️ **Build:** `Block(cfg)` with `ln_1`, `attn` (your `CausalSelfAttention`), `ln_2`, `mlp` (your `MLP`), and the two residual lines in `forward`.

In [ ]:
# ✍️ YOUR CODE: class Block(nn.Module)


In [ ]:
# ✅ CHECK Part 9
torch.manual_seed(0)
_b = Block(CFG).eval(); C = CFG.n_embd
_x = torch.randn(2, 10, C)
assert _b(_x).shape == _x.shape
_y = _x.clone(); _y[:, 6:] = torch.randn(2, 4, C)
assert torch.allclose(_b(_x)[:, :6], _b(_y)[:, :6], atol=1e-5), "the block must stay causal"
assert sum(isinstance(m, nn.LayerNorm) for m in _b.modules()) == 2, "two LayerNorms: ln_1 before attention, ln_2 before the MLP"
_n = sum(p.numel() for p in _b.parameters())
assert _n in (12 * C * C + 13 * C, 12 * C * C + 4 * C), f"expected {12*C*C + 13*C} params (attn + MLP + 2 LayerNorms); got {_n}"
# residual check: with every Linear zeroed, the block must be the identity
_z = Block(CFG).eval()
for m in _z.modules():
    if isinstance(m, nn.Linear):
        nn.init.zeros_(m.weight)
        if m.bias is not None:
            nn.init.zeros_(m.bias)
assert torch.allclose(_z(_x), _x), "missing residual: with zeroed sublayers, the block must return x unchanged"
print(f"✅ Part 9 passed: pre-norm residual block, {_n:,} params")

## Part 10 · Putting it together: the GPT (40 min)

🎯 **Goal:** the full model, which is `model.py`'s `GPT`.

```
idx (B, T) ──► token embedding (vocab → C)  +  position embedding (block_size → C)
            ──► dropout ──► Block × n_layer ──► final LayerNorm ──► lm_head (C → vocab) ──► logits (B, T, vocab)
```

🧠 **Idea.**
- **Token embedding** `nn.Embedding(vocab_size, C)`: a learned vector for each character, as in the bigram model, but C wide instead of 65.
- **Position embedding** `nn.Embedding(block_size, C)`: attention by itself has **no notion of order**. Shuffle the inputs and it gives the same set of outputs.
  So we add a learned vector for each position: `pos_emb(torch.arange(T, device=idx.device))`, shaped (T, C), broadcast over B. LLaMA uses RoPE instead; that's Monday.
- **lm_head** `nn.Linear(C, vocab_size)` turns each final token vector into 65 scores.
- **Loss:** as in the bigram model. Compute it on `logits.float()` (more accurate under bf16 later).
- **`generate(idx, max_new_tokens, temperature=1.0)`:** as in the bigram model, plus two things:
  - **crop** the input to the last `block_size` tokens, since the position table has no rows beyond that;
  - divide the last logits by **temperature** before the softmax. Below 1 the text is safer; above 1 it's wilder.

🛠️ **Build:** `GPT(cfg)`:
- `__init__`: the token and position embeddings, dropout, `n_layer` Blocks (`nn.Sequential(*[Block(cfg) for _ in range(cfg.n_layer)])`), `ln_f` and `lm_head`; store `cfg`.
- `forward(idx, targets=None)` returns `(logits, loss)`.
- `generate(...)` with `@torch.no_grad()`.

In [ ]:
# ✍️ YOUR CODE: class GPT(nn.Module)


In [ ]:
# ✅ CHECK Part 10  (= test_gpt_logits_and_loss, test_default_size_is_about_10m_params, test_generate)
torch.manual_seed(0)
_g = GPT(CFG)
_idx = torch.randint(0, CFG.vocab_size, (3, 12))
_logits, _loss = _g(_idx, _idx)
assert _logits.shape == (3, 12, CFG.vocab_size), f"logits should be (B, T, vocab) = (3, 12, 65); got {tuple(_logits.shape)}"
assert 3.5 < _loss.item() < 5.0, f"untrained loss should be near ln(65) = 4.17; got {_loss.item():.2f}"
assert _g(_idx)[1] is None
_n = sum(p.numel() for p in GPT(GPTConfig()).parameters())
assert 9e6 < _n < 12e6, f"the default config should have ~10.8M params; got {_n:,}"
_out = _g.eval().generate(torch.zeros(1, 1, dtype=torch.long), max_new_tokens=20)
assert _out.shape == (1, 21), "1 seed token + 20 new tokens"
assert _out.shape[1] > CFG.block_size, "generating past block_size works only if you crop the context"
print(f"✅ Part 10 passed: full GPT. Default size {_n:,} params (expected 10,795,841 with biases everywhere)")

## Part 11 · Training for real (45 min, of which the run is ~2–3 min)

🎯 **Goal:** train the default 10.8M-parameter GPT on the GPU to **val loss < 1.6**.

🧠 **Settings:** `block_size=256` (from the config), `batch_size=64`, `lr=1e-3`, **3000 steps**, evaluating every 250 steps.
On top of your Part 4 loop, add these four things:

| Addition | Code | Why |
|---|---|---|
| Weight decay | `AdamW(..., weight_decay=0.1)` | a mild pull of weights towards 0; regularises |
| LR warm-up + decay | `torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=3000, pct_start=0.05)`, and `sched.step()` after `opt.step()` | Adam's statistics start empty, so big early steps can blow up; warm up over the first 5%, then anneal down to settle into a minimum |
| Gradient clipping | `torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)` between `backward()` and `step()` | one bad batch can't throw the weights far off |
| bf16 autocast | `with torch.autocast(device, dtype=torch.bfloat16):` around the forward pass | matmuls in bfloat16 on the tensor cores: about 2× faster, half the memory. Weights stay FP32, and bf16 needs no loss scaling (unlike fp16) |

**Overfitting:** watch the two losses. Train keeps falling, but if val turns back **up**, the model has started memorising. 3000 steps is about where val bottoms out (~1.46).

🛠️ **Build:**
- `cfg = GPTConfig(vocab_size=vocab_size)`, `model = GPT(cfg).to(device)`, the optimiser and the scheduler.
- The loop, recording `history`: a list of `{"step": ..., "train": ..., "val": ...}` from `estimate_loss(model, 64, cfg.block_size, eval_iters=50)` at step 0, every 250 steps, and the final step.
- Print each entry and the elapsed time.

The training cell takes a few minutes. The ◼ square in VS Code shows it's still running.

In [ ]:
# ✍️ YOUR CODE: cfg, model, opt, sched, training loop -> history


In [ ]:
# Plot the learning curves
import matplotlib.pyplot as plt
steps = [h["step"] for h in history]
plt.figure(figsize=(7, 4))
plt.plot(steps, [h["train"] for h in history], label="train")
plt.plot(steps, [h["val"] for h in history], label="val")
plt.axhline(1.6, ls="--", c="gray", lw=1, label="target 1.6")
plt.axhline(math.log(65), ls=":", c="gray", lw=1, label="ln 65 (random)")
plt.xlabel("step"); plt.ylabel("cross-entropy loss"); plt.legend(); plt.grid(alpha=0.3)
plt.show()

In [ ]:
# ✅ CHECK Part 11
assert isinstance(history, list) and {"step", "train", "val"} <= set(history[-1]), "history: a list of dicts with step/train/val"
assert history[0]["val"] > 3.5, "step 0 should be measured before any training (~4.2–4.4)"
assert history[-1]["step"] == 3000
assert history[-1]["val"] < 1.6, f"val loss {history[-1]['val']:.3f} is not below 1.6 yet. Check warm-up, clipping and autocast"
print(f"✅ Part 11 passed: val loss {history[-1]['val']:.3f} (train {history[-1]['train']:.3f})")

## Part 12 · Sampling and the checkpoint (15 min)

🎯 **Goal:** read what your model writes, and save it for Monday's KV-cache work.

🛠️ **Build:**
1. `model.eval()`, then generate **500 characters** starting from id 0 (newline) at `temperature` 0.8, 1.0 and 1.3. Print each. What changes?
2. Save `sample.txt` (the 1.0 sample) next to this notebook.
3. Save the checkpoint: `torch.save({"model": model.state_dict(), "cfg": cfg, "vocab": vocab}, "checkpoints/tiny_gpt.pt")`, creating the folder first.
   Monday's code loads this file instead of retraining.

In [ ]:
# ✍️ YOUR CODE: samples at 3 temperatures, sample.txt, checkpoints/tiny_gpt.pt


In [ ]:
# ✅ CHECK Part 12
assert Path("sample.txt").exists() and len(Path("sample.txt").read_text()) >= 500, "write the 500-char sample to sample.txt"
_ck = torch.load("checkpoints/tiny_gpt.pt", weights_only=False)
assert {"model", "cfg", "vocab"} <= set(_ck), "the checkpoint needs model, cfg and vocab"
_re = GPT(_ck["cfg"]).to(device).eval(); _re.load_state_dict(_ck["model"])
_t = torch.randint(0, vocab_size, (1, 32), device=device)
assert torch.allclose(_re(_t)[0], model.eval()(_t)[0], atol=1e-5), "the reloaded model must give the same logits"
print("✅ Part 12 passed: checkpoint saved and reloads identically")

## Part 13 · From notebook to repo (30 min)

🎯 **Goal:** your notebook code becomes `model.py`, and the repo's own tests and training script pass.

🛠️ **Steps** (in a terminal, in this folder):
1. Open `model.py`. Replace each `raise NotImplementedError` / TODO with your code:
   - `CausalSelfAttention` from Part 7b;
   - `Block` from Part 9 (put your MLP inside it as `self.mlp = nn.Sequential(...)`, or add your `MLP` class above it);
   - `GPT.__init__` and `GPT.forward` from Part 10 (`generate` is already there).

   Keep the `GPTConfig` that's already in the file.
2. `pytest -q` should show **5 passed**. Compare each test in `test_model.py` with the notebook check it matches:

   | `test_model.py` | Notebook |
   |---|---|
   | `test_attention_shape` | 7b |
   | `test_attention_is_causal` | 6 (your own test) and 7b |
   | `test_gpt_logits_and_loss` | 10 |
   | `test_default_size_is_about_10m_params` | 10 |
   | `test_generate` | 10 |

3. `python train.py` should end with **val < 1.6** and print a sample (about 2 min).
4. Fill in the README's **Results** section: final val loss, time, and a few lines of `sample.txt`.
5. Commit on the `tiny-gpt` branch (one-line message), then merge into `main`.

💬 **Say these out loud** (interview practice):
- Why divide by √d? What goes wrong without it?
- What exactly does the causal mask block, and why is it needed in training but trivial when generating?
- Why pre-norm instead of post-norm? Why do residual connections make depth trainable?
- Why is the MLP 4× wide, and where are most of the parameters?
- Why does val loss rise again after about 3000 steps, and what are 3 ways to fight it?
- `generate` re-runs the whole window for every new token. What does that cost, and how does a KV cache fix it? (Monday.)